<a href="https://colab.research.google.com/github/marry12256/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26_week4_maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install transformers datasets sentencepiece accelerate -q

In [9]:
import os
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

In [10]:
csv_path = "/content/drive/MyDrive/final_labels.csv"

df = pd.read_csv(csv_path)

print(df.head())
print("Rows:", len(df))

                                               image  \
0  /content/drive/MyDrive/images/WhatsApp Image 2...   
1  /content/drive/MyDrive/images/WhatsApp Image 2...   
2  /content/drive/MyDrive/images/WhatsApp Image 2...   
3  /content/drive/MyDrive/images/WhatsApp Image 2...   
4  /content/drive/MyDrive/images/WhatsApp Image 2...   

                                             text  
0       کمرے کل بنے گا، انہیں مفت میں جانے دی گئی  
1  ہوٹل میں کھانا کھلا ہے۔ کھانا اچھا ہے۔ سٹاف کا  
2        لوگ دو ستارے ہیں۔ کمرے صاف اور روشن ہیں۔  
3       ہوٹل کا سٹاف اچھا میں ہے۔ ہمیں بہت انتظار  
4     کروایا۔ میں دوبارہ اس ہوٹل میں نہیں آؤں گا۔  
Rows: 200


In [11]:
missing = []

for p in df["image"]:
    if not os.path.exists(p):
        missing.append(p)

print("Missing:", len(missing))

Missing: 0


In [1]:
!pip install -U transformers sentencepiece tokenizers tiktoken
!pip uninstall -y transformers tokenizers sentencepiece tiktoken
!pip install transformers==4.44.2 tokenizers==0.19.1 sentencepiece tiktoken

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.9 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.1
    Uninstalling hf-xet-1.5.1:
      Successfully uninstalled hf-xet-1.5.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting

In [12]:
import transformers
import sentencepiece
import tokenizers

print("Transformers:", transformers.__version__)
print("SentencePiece OK")
print("Tokenizers OK")

Transformers: 4.44.2
SentencePiece OK
Tokenizers OK


In [13]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

print("Processor Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Processor Loaded Successfully


In [14]:
import torch
from transformers import VisionEncoderDecoderModel

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

# Move model to device
model.to(device)

print("Model Loaded on:", device)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model Loaded on: cuda


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

In [16]:
class UrduOCRDataset(Dataset):

    def __init__(self, dataframe, processor):
        self.dataframe = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.iloc[idx]["image"]
        text = str(self.dataframe.iloc[idx]["text"])

        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze()

        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [17]:
import pandas as pd

df = pd.read_csv(csv_path)

In [18]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [19]:
train_dataset = UrduOCRDataset(train_df, processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

In [20]:
val_dataset = UrduOCRDataset(val_df, processor)

test_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

print("Test Loader Created")

Test Loader Created


In [21]:
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

In [22]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id

In [23]:
import torch

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [24]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} complete | Average Loss: {avg_loss:.4f}"
    )

print("Training Complete")


Epoch 1/3
------------------------------
Batch 0/40 | Loss: 17.9508
Batch 10/40 | Loss: 5.6906
Batch 20/40 | Loss: 3.5971
Batch 30/40 | Loss: 3.2004
Epoch 1 complete | Average Loss: 4.5476

Epoch 2/3
------------------------------
Batch 0/40 | Loss: 2.6003
Batch 10/40 | Loss: 2.7818
Batch 20/40 | Loss: 2.6369
Batch 30/40 | Loss: 3.3148
Epoch 2 complete | Average Loss: 2.9887

Epoch 3/3
------------------------------
Batch 0/40 | Loss: 2.8406
Batch 10/40 | Loss: 3.3280
Batch 20/40 | Loss: 2.9703
Batch 30/40 | Loss: 2.7917
Epoch 3 complete | Average Loss: 2.7755
Training Complete


In [25]:
model.eval()

print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        # Generate predictions
        generated_ids = model.generate(pixel_values)

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        # Debug Output
        print("Generated IDs:", generated_ids)
        print("Generated Text:", generated_text)
        print("Actual Text:", actual_text)
        print("-" * 50)

        for pred, actual in zip(generated_text, actual_text):
            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print(f"Predicted: {pred}")
            print(f"Actual:    {actual}")
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Generated IDs: tensor([[    2,     0, 44148, 44148, 44148,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1],
        [    2,     0, 44148, 44148, 44148,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1],
        [    2,     0, 44148, 44148, 44148, 44148, 44148, 44148, 44148, 44148,
         44148, 44148, 44148, 44148,     1,     1,     1,     1,     1,     1],
        [    2,     0, 44148, 44148, 44148,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1]],
       device='cuda:0')
Generated Text: ['���', '���', '������������', '���']
Actual Text: ['ہوٹل کا ڈینٹ صفائی سے ہے۔ ہوٹل ٹھیک ہے۔', 'تعلیم ہر انسان کا حق ہے', 'اقدامات اور محاذ جنگ پر کارکردگی کا تجزیہ اور بہتر', 'خاموش رہو']
--------------------------------------------------
Predicted: ���
Actual:    ہوٹل کا ڈینٹ صفائی سے ہے۔ ہوٹل ٹھیک ہے۔

P

In [26]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining


My model accuracy is 0.0%
Training loss went from 4.5476 to 2.7755.